In [1]:
import os
import math
import pandas as pd
from tqdm import tqdm
from scapy.all import rdpcap, IP, TCP, UDP
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import classification_report


calculate_entropy(payload_bytes)
作用：
计算给定 payload 的 Shannon 熵，用于衡量内容的“复杂度”（比如加密数据 vs 明文数据）。

用途：
作为 payload-based 特征之一，用于辅助识别设备通信行为的差异。

如果某设备主要使用：

TCP、DNS、NTP；

Payload 多为明文、短小；

TCP window 很小；

它就可能是一个简单的传感器或小型控制器。

而如果某设备：

使用 TCP+HTTP/HTTPS，Payload 很长；

熵高（说明是加密数据）；

window size 大；

它可能是摄像头、音响、智能助手等大数据量设备。

In [2]:
def calculate_entropy(payload_bytes):
    if not payload_bytes:
        return 0
    byte_freq = [0] * 256
    for b in payload_bytes:
        byte_freq[b] += 1
    entropy = 0
    for f in byte_freq:
        if f > 0:
            p = f / len(payload_bytes)
            entropy -= p * math.log(p, 2)
    return entropy




extract_packet_features(pkt)
    作用：
    从一个单独的网络包中提取 20 个特征，包括：

    17 个协议相关的 header 特征（如是否是 TCP/UDP/DNS/...）

    3 个 payload 特征：熵、payload 长度、TCP 窗口大小

    输出：该包的特征向量（list of 20 values）

In [3]:
def extract_packet_features(pkt):
    features = []

    # Packet Header (17 features simplified)
    features.append(int(pkt.haslayer("TCP")))
    features.append(int(pkt.haslayer("UDP")))
    features.append(int(pkt.haslayer("DNS")))
    features.append(int(pkt.haslayer("MDNS")))
    features.append(int(pkt.haslayer("HTTP")))
    features.append(int(pkt.haslayer("ICMP")))
    features.append(int(pkt.haslayer("ARP")))
    features.append(int(pkt.haslayer("BOOTP")))
    features.append(int(pkt.haslayer("NTP")))
    features.append(int(pkt.haslayer("EAPOL")))
    features.append(int(pkt.haslayer("IP")))
    features.append(1 if pkt.haslayer("IP") and pkt["IP"].flags == 2 else 0)
    features.append(int(pkt.haslayer("IPv6")))
    features.append(int(pkt.haslayer("SSDP")))
    features.append(int(pkt.haslayer("Padding")))
    features.append(int(pkt.haslayer("Raw")))
    features.append(1)  # Dummy for 17th feature placeholder

    # Payload-based features
    try:
        payload = bytes(pkt.payload.payload.payload)
    except:
        payload = b''
    entropy = calculate_entropy(payload)
    payload_len = len(payload)
    tcp_win = pkt[TCP].window if pkt.haslayer("TCP") else 0
    features.extend([entropy, payload_len, tcp_win])
    return features

make up label

In [4]:
# def simplify_label(label):
#     if "audio" in label:
#         return "audio_control"
#     elif "voice" in label:
#         return "voice"
#     elif "volume" in label:
#         return "volume"
#     elif "power" in label:
#         return "power"
#     else:
#         return label

In [6]:
import os

def collect_pcap_files(root_dir):
    """
    递归遍历任意层级目录，收集所有 .pcap 文件，label 为路径中倒数两级目录拼接
    """
    pcap_label_list = []
    for dirpath, _, filenames in os.walk(root_dir):
        for file in filenames:
            if file.endswith(".pcap"):
                full_path = os.path.join(dirpath, file)
                parts = os.path.normpath(full_path).split(os.sep)
                label = "_".join(parts[-3:-1]) if len(parts) >= 3 else os.path.basename(dirpath)
                pcap_label_list.append((full_path, label))
    return pcap_label_list


处理一个或多个PCAP file

In [5]:
def process_pcap_file(pcap_path, label="Device_X", max_packets=2000, group_size=5):
    packets = rdpcap(pcap_path, count=max_packets)
    filtered = [pkt for pkt in packets if pkt.haslayer("IP") and (pkt.haslayer("TCP") or pkt.haslayer("UDP"))]
    sessions = [filtered[i:i+group_size] for i in range(0, len(filtered) - group_size + 1)]  # 每个 session = 5 个包
    fingerprints = []
    for session in tqdm(sessions, desc=f"Processing {os.path.basename(pcap_path)}"):
        combined = []
        for pkt in session:   
            combined.extend(extract_packet_features(pkt))  # 每包提取 20 个特征
        if len(combined) == group_size * 20:   #verify the each session with 5*20=100 features; 指纹100维
            fingerprints.append(combined)
    df = pd.DataFrame(fingerprints)
    df['label'] = label
    return df

def process_multiple_pcaps(pcap_label_list, output_path="all_features.csv"):
    all_data = []
    for path, label in pcap_label_list:
        df = process_pcap_file(path, label)
        all_data.append(df)
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df.to_csv(output_path, index=False)
    print(f"✅ All fingerprints saved to: {output_path}")
    return combined_df



读取所有pcap files

Train model

In [7]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def train_and_evaluate_model(csv_path, sample_size=5000):
    print(f"\n📊 Training model on: {csv_path} (sample {sample_size} rows)\n")
    data = pd.read_csv(csv_path)
    if sample_size < len(data):
        data = data.sample(n=sample_size, random_state=42)

    X = data.drop("label", axis=1)
    y = data["label"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    print("\n🔍 Classification Report:\n")
    print(classification_report(y_test, y_pred, zero_division=0))
    return clf, y_test, y_pred 

In [8]:

def filter_by_label_count(csv_path, min_count=100, output_path="filtered_iot.csv"):
    """
    Keep only labels that have at least `min_count` samples.
    """
    df = pd.read_csv(csv_path)
    label_counts = df['label'].value_counts()
    valid_labels = label_counts[label_counts >= min_count].index
    df_filtered = df[df['label'].isin(valid_labels)]
    df_filtered.to_csv(output_path, index=False)
    print(f"✅ Retained {len(valid_labels)} labels with ≥{min_count} samples each.")
    print(f"🔢 Total samples after filtering: {len(df_filtered)}")
    return output_path


In [ ]:

def filter_top_labels(csv_path, top_n=10):
    df = pd.read_csv(csv_path)
    top_labels = df['label'].value_counts().head(top_n).index
    df_top = df[df['label'].isin(top_labels)]
    df_top.to_csv("top_labels.csv", index=False)
    print(f"✅ Saved top {top_n} labels to top_labels.csv with {len(df_top)} samples.")
    print(f"📋 Labels used: {list(top_labels)}")
    return "top_labels.csv"



✅ Retained 347 labels with ≥100 samples each.
🔢 Total samples after filtering: 3771758
✅ Saved top 12 labels to top_labels.csv with 876776 samples.
📋 Labels used: ['yi-camera_android_lan_recording', 'yi-camera_android_lan_photo', 'ring-doorbell_android_wan_watch', 'yi-camera_android_wan_recording', 'ring-doorbell_android_lan_watch', 'ring-doorbell_alexa_stop', 'microseven-camera_android_lan_watch', 'ring-doorbell_alexa_watch', 'wansview-cam-wired_android_wan_recording', 'lefun-cam-wired_android_lan_watch', 'luohe-spycam_android_lan_photo', 'lefun-cam-wired_android_lan_photo']

📊 Training model on: top_labels.csv (sample 10000 rows)


🔍 Classification Report:

                                          precision    recall  f1-score   support

       lefun-cam-wired_android_lan_photo       0.65      0.75      0.70       146
       lefun-cam-wired_android_lan_watch       0.68      0.54      0.60       141
          luohe-spycam_android_lan_photo       0.80      0.70      0.75       153
   

In [10]:
def extract_device_label(label):
    return label.split("_")[0]


In [11]:

def run_iot_pipeline(
    csv_path,
    top_n_labels=12,
    sample_size=10000,
    group_size=5,
    model_type="rf"
):
    print(f"📁 Loading data from: {csv_path}")
    df = pd.read_csv(csv_path)

    # Step 1: Simplify labels to device only
    df["label"] = df["label"].apply(extract_device_label)

    # Step 2: Filter top N labels
    top_labels = df["label"].value_counts().head(top_n_labels).index
    df = df[df["label"].isin(top_labels)]
    print(f"✅ Filtered to top {top_n_labels} devices. Samples: {len(df)}")

    # Step 3: Sample
    if sample_size < len(df):
        df = df.sample(n=sample_size, random_state=42)

    # Step 4: Feature / Label split
    X = df.drop("label", axis=1)
    y = df["label"]

    # Step 5: Stratified Split
    from sklearn.model_selection import StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_idx, test_idx in sss.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Step 6: Train model
    if model_type == "rf":
        from sklearn.ensemble import RandomForestClassifier
        clf = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42)
    elif model_type == "lgbm":
        from lightgbm import LGBMClassifier
        clf = LGBMClassifier(n_estimators=100, max_depth=16, class_weight='balanced')
    else:
        raise ValueError("Unsupported model type")

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Step 7: Report
    from sklearn.metrics import classification_report
    print("\n🔍 Classification Report (device-level):\n")
    print(classification_report(y_test, y_pred, zero_division=0))

    return clf, y_test, y_pred


In [18]:
root_dir = r"C:\Users\Martin Lin\Desktop\SE6014\project\dataset\data_1\iot-data\uk"
pcaps = collect_pcap_files(root_dir)



# 生成特征文件
df_all = process_multiple_pcaps(pcaps, output_path="iot_all_devices.csv")




Processing 2019-05-04_14_45_21.27s.pcap: 100%|██████████| 647/647 [00:00<00:00, 1470.46it/s]
Processing 2019-05-04_15_02_37.28s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_15_19_53.27s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_16_11_41.28s.pcap: 100%|██████████| 716/716 [00:00<00:00, 1482.40it/s]
Processing 2019-05-04_16_28_57.28s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_16_46_14.28s.pcap: 100%|██████████| 667/667 [00:00<00:00, 1456.58it/s]
Processing 2019-05-04_17_03_28.27s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_17_20_41.27s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_17_37_56.28s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_17_55_15.27s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_18_12_30.27s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_18_47_01.27s.pcap: 100%|██████████| 1142/1142 [00:00<00:00, 1380.85it/s]
Processing 2019-05-04_19_04_19.27s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_19_21_36.27s.pcap: 0it [00:00, ?it/s]
Processing 2019-05-04_19_3

✅ All fingerprints saved to: iot_all_devices.csv


In [16]:
filtered_csv = filter_by_label_count("iot_all_devices.csv", min_count=100)
top_csv = filter_top_labels("filtered_iot.csv", top_n=12)
trained_model = train_and_evaluate_model(top_csv, sample_size=10000)


✅ Retained 347 labels with ≥100 samples each.
🔢 Total samples after filtering: 3771758
✅ Saved top 12 labels to top_labels.csv with 876776 samples.
📋 Labels used: ['yi-camera_android_lan_recording', 'yi-camera_android_lan_photo', 'ring-doorbell_android_wan_watch', 'yi-camera_android_wan_recording', 'ring-doorbell_android_lan_watch', 'ring-doorbell_alexa_stop', 'microseven-camera_android_lan_watch', 'ring-doorbell_alexa_watch', 'wansview-cam-wired_android_wan_recording', 'lefun-cam-wired_android_lan_watch', 'luohe-spycam_android_lan_photo', 'lefun-cam-wired_android_lan_photo']

📊 Training model on: top_labels.csv (sample 10000 rows)


🔍 Classification Report:

                                          precision    recall  f1-score   support

       lefun-cam-wired_android_lan_photo       0.65      0.75      0.70       146
       lefun-cam-wired_android_lan_watch       0.68      0.54      0.60       141
          luohe-spycam_android_lan_photo       0.80      0.70      0.75       153
   

In [17]:
clf, y_test, y_pred = run_iot_pipeline(
    csv_path="filtered_iot.csv",     # 你已经预处理好的特征 CSV
    top_n_labels=12,                 # 对齐论文，选 12 个设备
    sample_size=10000,              # 训练样本总数
    model_type="rf"                 # 可选 "rf" 或 "lgbm"
)

📁 Loading data from: filtered_iot.csv
✅ Filtered to top 12 devices. Samples: 3138533

🔍 Classification Report (device-level):

                    precision    recall  f1-score   support

 amcrest-cam-wired       0.99      0.99      0.99       248
          cloudcam       0.97      0.96      0.96       134
            fridge       0.81      0.93      0.87        76
   lefun-cam-wired       0.94      0.94      0.94       213
        lgtv-wired       0.70      0.72      0.71        46
      luohe-spycam       0.81      0.82      0.82       233
     ring-doorbell       0.99      0.99      0.99       183
   samsungtv-wired       0.64      0.59      0.62        49
     t-philips-hub       0.93      0.94      0.94       163
wansview-cam-wired       0.97      0.93      0.95       249
         yi-camera       0.84      0.85      0.85       259
    zmodo-doorbell       0.98      0.94      0.96       147

          accuracy                           0.91      2000
         macro avg       0.88  